# Week 2: From Experiments to Observational Data

This notebook follows the Week 2 page, in the same order:
<https://yoonismo.github.io/causal-computing/week2.html>

Read the explanations on the page and run the cells here **in order** from top to bottom. Some cells ask you to write code yourself: they are marked `# TODO`, and each is followed by a **check** cell. Write your answers to the questions in the empty text cells.

## 0. Before you start

- Check **Select Kernel** at the top right: it should say **ECON 481 (Python 3.12)**.
- If a cell fails with `ModuleNotFoundError`, a package is missing from your environment. See `README.md`.
- The data sets are downloaded from the internet when you load them, so you need a connection.
- A check cell prints `not done yet` until you fill in the `# TODO` cell above it. Every cell must still run from top to bottom when you push: GitHub runs this notebook after every push.

## Lab 1 — A world where we know the truth

### 1.1 Create both potential outcomes

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy.stats import norm

import matplotlib.pyplot as plt

rng = np.random.default_rng(481)
n = 1000

y0 = rng.normal(loc=10, scale=3, size=n)       # Y(0): outcome without treatment
tau = 2                                        # the treatment effect, the same for everyone
y1 = y0 + tau                                  # Y(1): outcome with treatment

print("True ATE:", (y1 - y0).mean())

**Question 1.** Why can we compute the true ATE here, but never in real data?

*Your answer to Question 1:*

### 1.2 Randomize once

In [ ]:
d = rng.permutation(np.repeat([1, 0], n // 2))   # half treated, at random
y = d * y1 + (1 - d) * y0                         # what we would observe

print("Difference in means:", (y[d == 1].mean() - y[d == 0].mean()).round(3))

**Question 2.** The estimate is not exactly the true ATE. Is the estimator wrong? (The optional section at the end lets you check.)

*Your answer to Question 2:*

## Lab 2 — A real experiment: the National Supported Work Demonstration

Load the NSW data, then split it into the treated and the control group.

In [ ]:
from causaldata import nsw_mixtape

nsw = nsw_mixtape.load_pandas().data

In [ ]:
treated = nsw[nsw["treat"] == 1]
control = nsw[nsw["treat"] == 0]

Group sizes: $n_1$ for the treated group, $n_0$ for the control group, and $n = n_1 + n_0$. The whole sample size is called `n_nsw`, because `n` already holds the size of the simulated world in Lab 1.

In [ ]:
n_nsw = nsw.shape[0]                       # the entire sample size
n_1 = len(nsw[nsw["treat"] == 1])          # the number of treated units
n_0 = len(nsw[nsw["treat"] == 0])          # the number of control units

print("the number of treated:", n_1, "  the number of control:", n_0)

Group means $\overline{Y}_1$ and $\overline{Y}_0$: the sum of the outcomes in each group divided by the number of people in it. Their difference is the estimate $\hat{\tau}_{\text{ATE}} = \overline{Y}_1 - \overline{Y}_0$.

In [ ]:
ybar_1 = treated["re78"].sum() / n_1   # mean outcome of the treated group
ybar_0 = control["re78"].sum() / n_0   # mean outcome of the control group

diff = ybar_1 - ybar_0                 # the estimate of the ATE
diff

Next, the standard error of $\hat{\tau}_{\text{ATE}}$. Let $s_1^2$ and $s_0^2$ be the sample variances of the outcome $Y_i$ within the treated and the control group:

$$
s_1^2 = \frac{1}{n_1 - 1} \sum_{i \in \text{treated}} (Y_i - \overline{Y}_1)^2, \qquad
s_0^2 = \frac{1}{n_0 - 1} \sum_{i \in \text{control}} (Y_i - \overline{Y}_0)^2 .
$$

Dividing each by its group size gives the estimated variance of each group mean, $s_1^2 / n_1$ and $s_0^2 / n_0$. The standard error of $\hat{\tau}_{\text{ATE}}$ is the square root of their sum:

$$
\text{SE}_{\text{ATE}} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_0^2}{n_0}} .
$$

In [ ]:
s2_1 = np.sum((treated["re78"] - ybar_1) ** 2) / (n_1 - 1)   # sample variance in the treated group
s2_0 = np.sum((control["re78"] - ybar_0) ** 2) / (n_0 - 1)   # sample variance in the control group

se = np.sqrt(s2_1 / n_1 + s2_0 / n_0)

The 95% confidence interval is

$$
\text{CI}_{95} = \left[\hat{\tau}_{\text{ATE}} - C_{0.975} \cdot \text{SE}_{\text{ATE}},\; \hat{\tau}_{\text{ATE}} + C_{0.975} \cdot \text{SE}_{\text{ATE}}\right],
$$

where $C_{0.975} \approx 1.96$ is the 97.5% quantile of the standard normal distribution. The cell below computes it.

In [ ]:
quantile_975 = norm.ppf(0.975)   # C_0.975
quantile_975

**Your turn.** Compute the lower and upper ends of the 95% confidence interval, using `diff`, `se` and `quantile_975`.

In [ ]:
# TODO: replace each None
ci_low = None
ci_high = None
print("95% interval:", ci_low, "to", ci_high)

In [ ]:
# Check (do not edit)
if ci_low is None or ci_high is None:
    print("not done yet")
else:
    ok = np.isclose(ci_low, diff - quantile_975 * se) and np.isclose(ci_high, diff + quantile_975 * se)
    print("matches" if ok else "does not match: check the formula above")

### The same steps in a few lines of `pandas`

**Your turn.** Read the section "In short, we can do all the previous code work as below" on the Week 2 page. Compute the difference in means and the standard error again, this time with the `pandas` methods `.mean()` and `.var()`. Do you get the same numbers as above?

In [ ]:
# TODO: replace each None
diff_pandas = None   # difference in means, with .mean()
se_pandas = None     # standard error, with .var() and len()
print("Difference in means:", diff_pandas)
print("Standard error:     ", se_pandas)

In [ ]:
# Check (do not edit)
if diff_pandas is None or se_pandas is None:
    print("not done yet")
else:
    ok = np.isclose(diff_pandas, diff) and np.isclose(se_pandas, se)
    print("same as the step-by-step numbers" if ok else "different: compare with the page")

### The same numbers in one line

A regression of `re78` on `treat` with `cov_type="HC2"` gives the same estimate and standard error.

**Your turn.** Copy the regression from the section "The same numbers in one line" on the Week 2 page, run it, and compare the coefficient on `treat` and its standard error with your numbers above.

In [ ]:
# TODO: replace None with the fitted regression (do not forget the import)
model = None

if model is not None:
    print("Difference in means:", model.params["treat"].round(3))
    print("Standard error:     ", model.bse["treat"].round(3))
    print("95% interval:       ", model.conf_int().loc["treat"].round(3).tolist())

In [ ]:
# Check (do not edit)
if model is None:
    print("not done yet")
else:
    ok = np.isclose(model.params["treat"], diff) and np.isclose(model.bse["treat"], se)
    print("same as the step-by-step numbers" if ok else "different: did you use cov_type=\"HC2\"?")

### Balance check

Randomization should make the groups similar **before** treatment. For each covariate $X$, the table shows the group means $\overline{X}_1$ and $\overline{X}_0$ and the **standardized difference**

$$
\text{std. diff.} = \frac{\overline{X}_1 - \overline{X}_0}{\sqrt{(s_{X,1}^2 + s_{X,0}^2) / 2}},
$$

where $s_{X,1}^2$ and $s_{X,0}^2$ are the sample variances of $X$ in each group. Dividing by a standard deviation removes the units, so a variable in dollars (`re74`) and a 0/1 variable (`black`) can be compared on the same scale.

In [ ]:
covariates = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

balance = pd.DataFrame({
    "treated": treated[covariates].mean(),
    "control": control[covariates].mean(),
})
balance["std. diff."] = (balance["treated"] - balance["control"]) / np.sqrt(
    (treated[covariates].var() + control[covariates].var()) / 2
)
print(balance.round(3))

Comparing the mean outcomes of two groups is a fair comparison only when the two groups are similar, on average, in their characteristics before treatment. Random assignment makes them similar on average; in any one sample, some differences remain by chance.

## Lab 3 — Observational studies

### 3.2 A simulation: let people choose

In [ ]:
p = 1 / (1 + np.exp((y0 - 10) / 2))     # lower y0 -> higher chance of treatment
d_sel = rng.binomial(1, p)
y_sel = d_sel * y1 + (1 - d_sel) * y0

naive = y_sel[d_sel == 1].mean() - y_sel[d_sel == 0].mean()
att = (y1 - y0)[d_sel == 1].mean()
selection_bias = y0[d_sel == 1].mean() - y0[d_sel == 0].mean()

print("Naive difference:", naive.round(3))
print("ATT:             ", att.round(3))
print("Selection bias:  ", selection_bias.round(3))
print("ATT + bias:      ", (att + selection_bias).round(3))

**Question 3.** The naive difference is negative although the effect is exactly 2 for everyone. Which line explains why? Check that the last two lines add up.

*Your answer to Question 3:*

**Question 4.** In this simulation, people with a *lower* outcome without treatment are more likely to be treated. Give one real-world example where this happens.

*Your answer to Question 4:*

### 3.3 LaLonde (1986): swap the control group

In [ ]:
from causaldata import cps_mixtape

cps = cps_mixtape.load_pandas().data
print("Treated minus CPS:", (treated["re78"].mean() - cps["re78"].mean()).round(3))

In [ ]:
pd.DataFrame({
    "NSW treated": treated[covariates].mean(),
    "NSW control": control[covariates].mean(),
    "CPS": cps[covariates].mean(),
}).round(3)

**Question 5.** Taking 1,794 as the ATT, how large is the selection bias? Which rows of the table explain its sign?

*Your answer to Question 5:*

### 3.4 What if we keep only CPS men who look like the trainees?

In [ ]:
groups = {
    "all CPS": cps,
    "CPS, re75 below 5,000": cps[cps["re75"] < 5000],
    "CPS, no earnings in 1974 and 1975": cps[(cps["re74"] == 0) & (cps["re75"] == 0)],
}
for label, comparison in groups.items():
    gap = treated["re78"].mean() - comparison["re78"].mean()
    print(f"{label:35s} n = {len(comparison):6d}   gap = {gap:9.1f}")

**Question 6.** The gap moves a lot depending on the rule we use to choose comparison people. What should a good rule be based on, and why?

*Your answer to Question 6:*

## Optional: randomize many times

In [ ]:
reps = 2000
D = rng.permuted(np.tile(np.repeat([1, 0], n // 2), (reps, 1)), axis=1)
Y = D * y1 + (1 - D) * y0

estimates = (Y * D).sum(axis=1) / D.sum(axis=1) - (Y * (1 - D)).sum(axis=1) / (1 - D).sum(axis=1)
print("Mean of the 2,000 estimates:", (np.sum(estimates) / reps).round(3))
plt.hist(estimates, bins=40, color="lightblue", edgecolor="white")
plt.axvline(tau, color="red", label="true ATE")
plt.axvline(np.sum(estimates) / reps, color="blue", linestyle="--", label="mean of the estimates")
plt.legend()
plt.show()

**Optional question.** Where is the centre of the histogram? What does that tell you about the difference in means under random assignment?

*Your answer to the optional question:*

## Next

Open `star_practice.ipynb` and repeat Labs 2 and 3 on your own with the Tennessee STAR experiment.